# 03 — Customer-Level Feature Engineering, RFM Analysis & Preprocessing

**Author:** Person B (RFM & Customer Segmentation Lead)  
**Project:** Online Retail Customer Analytics & Segmentation  
**Upstream Input:** `data/interim_structural.parquet` (Cleaned transaction dataset from Notebook 01)  
**Downstream Output:** `data/rfm_features.parquet` (Scored RFM) & `data/rfm_preprocessed.parquet` (Log-Scaled features for clustering)

## 1. Business & Technical Context

### 1.1 From Transactions to Customer Entities
In transactional retail data, each row represents a line item within a basket. However, strategic marketing actions, loyalty rewards, and retention campaigns target **customers**, not line items or individual baskets. Customer-level feature engineering aggregates fine-grained purchase histories into entity-level metrics that describe customer value, engagement, and retention risk.

### 1.2 The Mechanics of RFM Analysis
RFM is a classic behavioral framework that translates historical transaction logs into three core dimensions:
- **Recency (R):** Days elapsed between the reference cutoff date and the customer's most recent purchase invoice. Lower recency indicates active engagement.
- **Frequency (F):** Count of unique purchase invoices (orders) completed by the customer. Higher frequency indicates habitual purchasing behavior.
- **Monetary Value (M):** Total net sales revenue (£) generated by the customer (net of item returns and cancellations). Higher monetary value indicates commercial significance.

### 1.3 Strategic Business Value
1. **Actionable Personalization:** Prevents over-discounting loyal buyers while targeting aggressive retention offers precisely to churning users.
2. **Early Warning Churn Indicators:** Detects declining Recency before accounts are permanently lost.
3. **Protecting Core Revenue:** Distinguishes high-value wholesale buyers from small-batch retail buyers.

In [1]:
import os
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['OMP_NUM_THREADS'] = '1'

import sys
sys.path.append('../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from data_cleaning import load_cleaned_data, get_identified_customer_transactions
from rfm_analysis import calculate_rfm, score_rfm, preprocess_rfm_for_clustering

# Styling configuration
sns.set_theme(style="whitegrid")
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.edgecolor'] = '#cccccc'
plt.rcParams['axes.linewidth'] = 0.8
fig_dir = Path('../reports/figures')
fig_dir.mkdir(parents=True, exist_ok=True)

print("Environment setup complete and custom modules imported from /src.")

Environment setup complete and custom modules imported from /src.


## 2. Dataset Loading & Data Audit

We load `data/interim_structural.parquet` produced by Notebook 01. We explicitly document and isolate missing `CustomerID` rows (guest checkout blocks) before creating customer-level metrics.

In [2]:
df_raw = load_cleaned_data('../data/interim_structural.parquet')
print(f"Total transaction rows loaded: {len(df_raw):,}")

# Identified vs Guest Checkout Analysis
df_known = get_identified_customer_transactions(df_raw)
n_known = len(df_known)
n_guest = len(df_raw) - n_known

raw_line_rev = df_raw['Quantity'] * df_raw['UnitPrice']
guest_rev = raw_line_rev[df_raw['CustomerID'].isna()].sum()
total_rev = raw_line_rev.sum()

print(f"Identified customer rows: {n_known:,} ({n_known/len(df_raw):.1%})")
print(f"Guest checkout rows     : {n_guest:,} ({n_guest/len(df_raw):.1%})")
print(f"Identified net revenue  : £{total_rev - guest_rev:,.2f} ({(total_rev - guest_rev)/total_rev:.1%} of total revenue)")
print(f"Guest checkout revenue  : £{guest_rev:,.2f} ({guest_rev/total_rev:.1%} of total revenue)")

Total transaction rows loaded: 536,641
Identified customer rows: 401,604 (74.8%)
Guest checkout rows     : 135,037 (25.2%)
Identified net revenue  : £8,278,519.42 (85.1% of total revenue)
Guest checkout revenue  : £1,447,487.53 (14.9% of total revenue)


### Business Decision & Audit Log: Handling Missing CustomerIDs
> **Finding & Decision:**
> - Missing `CustomerID` records represent whole unlinked transactions (guest checkout blocks), accounting for **14.9% of net revenue** (£1.44M) and **25.2% of transaction lines**.
> - Because guest orders lack a persistent customer tracking ID across visits, they cannot be aggregated into customer entities.
> - **Action:** Guest transactions are excluded from customer-level RFM modeling and algorithmic clustering, but their revenue share is explicitly acknowledged in executive summaries and business limitations.

## 3. Customer-Level Feature Engineering

We establish the reference date as `max(InvoiceDate) + 1 day` (`2011-12-10 12:50:00`) to compute:
- **Recency:** `(ReferenceDate - max(InvoiceDate)).days`
- **Frequency:** Count of unique purchase invoices (`InvoiceNo` not starting with `'C'`)
- **Monetary:** Sum of `Quantity * UnitPrice` (net of returned items)
- **AvgOrderValue (AOV):** `Monetary / Frequency`
- **TenureDays:** `(ReferenceDate - min(InvoiceDate)).days`

In [3]:
ref_date = df_known['InvoiceDate'].max() + pd.Timedelta(days=1)
print(f"Dataset Cutoff Date : {df_known['InvoiceDate'].max()}")
print(f"RFM Reference Date : {ref_date}")

rfm_raw = calculate_rfm(df_known, reference_date=ref_date)
print(f"Total unique identified customers: {len(rfm_raw):,}")

print("\n--- Summary Statistics of Raw Customer Metrics ---")
print(rfm_raw[['Recency', 'Frequency', 'Monetary', 'AvgOrderValue', 'TenureDays']].describe())

Dataset Cutoff Date : 2011-12-09 12:50:00
RFM Reference Date : 2011-12-10 12:50:00
Total unique identified customers: 4,372

--- Summary Statistics of Raw Customer Metrics ---
           Recency    Frequency       Monetary  AvgOrderValue   TenureDays
count  4372.000000  4372.000000    4372.000000    4372.000000  4372.000000
mean     92.047118     4.247255    1893.531433     358.823808   225.779048
std     100.765435     7.681573    8218.696204     409.249253   118.412473
min       1.000000     1.000000   -4287.630000   -4287.630000     1.000000
25%      17.000000     1.000000     291.795000     170.900000   115.000000
50%      50.000000     2.000000     644.070000     277.700000   253.000000
75%     143.000000     5.000000    1608.335000     419.765000   332.000000
max     374.000000   210.000000  279489.020000    7178.630000   374.000000


### Audit of Customer Net Spending & Return Anomalies
We inspect customers with `Monetary <= 0` (where returns/cancellations exceed historical purchases within the dataset window).

In [4]:
neg_m = rfm_raw[rfm_raw['Monetary'] <= 0]
print(f"Customers with Net Monetary <= 0: {len(neg_m)} ({len(neg_m)/len(rfm_raw):.2%})")
if len(neg_m) > 0:
    print(neg_m[['CustomerID', 'LastInvoice', 'TotalInvoices', 'PurchaseInvoices', 'CancellationInvoices', 'Monetary']].head(10))

Customers with Net Monetary <= 0: 50 (1.14%)
     CustomerID         LastInvoice  TotalInvoices  PurchaseInvoices  \
0         12346 2011-01-18 10:17:00              2                 1   
125       12503 2011-01-06 11:51:00              1                 0   
127       12505 2011-02-11 13:39:00              1                 0   
212       12605 2010-12-09 15:41:00              1                 0   
264       12666 2010-12-15 14:42:00              1                 0   
415       12870 2010-12-08 14:27:00              1                 0   
467       12943 2011-02-11 15:13:00              1                 0   
619       13154 2011-07-18 15:21:00              1                 0   
692       13256 2011-11-25 15:57:00              1                 1   
775       13364 2011-10-04 09:48:00              2                 1   

     CancellationInvoices  Monetary  
0                       1      0.00  
125                     1  -1126.00  
127                     1     -4.50  
212       

**Interpretation of Negative Net Monetary Spenders:**
- 50 out of 4,372 customers (1.14%) have net zero or negative spending.
- These reflect accounts where large cancellation entries occurred without matching prior purchase records in the 1-year data window (e.g. customer 12346 returned £74,215 worth of goods).
- **Handling:** For quantile RFM scoring, these are scored naturally in the lowest Monetary quantile. For log-transformation and clustering, customers with `Monetary > 0` are processed to prevent undefined logarithm errors (`log(<=0)`).

## 4. The Mechanics of RFM Quantile Scoring

We divide RFM metrics into 5 equal quantiles using `pd.qcut` (scoring 1 to 5):
- **Recency (R):** Inverted scoring — **5 = Most Recent** (lowest days since purchase), **1 = Least Recent** (dormant/churned).
- **Frequency (F):** Direct scoring — **5 = Highest Order Count**, **1 = Lowest Order Count**. (Rank ties resolved via `rank(method='first')`).
- **Monetary (M):** Direct scoring — **5 = Highest Net Revenue**, **1 = Lowest Net Revenue**.

`RFM_Segment` is concatenated as a 3-character string (e.g., `'555'`), and `RFM_Score` is the integer sum ($R + F + M \in [3, 15]$).

In [5]:
rfm_scored = score_rfm(rfm_raw)

print("RFM Quantile Scoring Completed. Sample scored records:")
print(rfm_scored[['CustomerID', 'Recency', 'R_Score', 'Frequency', 'F_Score', 'Monetary', 'M_Score', 'RFM_Segment', 'RFM_Score']].head(10))

RFM Quantile Scoring Completed. Sample scored records:
   CustomerID  Recency  R_Score  Frequency  F_Score  Monetary  M_Score  \
0       12346      326        1          1        1      0.00        1   
1       12347        2        5          7        5   4310.00        5   
2       12348       75        2          4        4   1797.24        4   
3       12349       19        4          1        1   1757.55        4   
4       12350      310        1          1        1    334.40        2   
5       12352       36        3          8        5   1545.41        4   
6       12353      204        1          1        1     89.00        1   
7       12354      232        1          1        1   1079.40        4   
8       12355      214        1          1        1    459.40        2   
9       12356       23        4          3        3   2811.43        5   

  RFM_Segment  RFM_Score  
0         111          3  
1         555         15  
2         244         10  
3         414         

### Quantile Cutoff Thresholds

In [6]:
print("--- Recency Score Cutoffs (Days) ---")
r_cutoffs = pd.qcut(rfm_scored['Recency'], q=5, labels=[5, 4, 3, 2, 1])
r_summary = rfm_scored.groupby(r_cutoffs, observed=False)['Recency'].agg(['min', 'max', 'count'])
print(r_summary)

print("\n--- Frequency Score Cutoffs (Orders) ---")
f_cutoffs = pd.qcut(rfm_scored['Frequency'].rank(method='first'), q=5, labels=[1, 2, 3, 4, 5])
f_summary = rfm_scored.groupby(f_cutoffs, observed=False)['Frequency'].agg(['min', 'max', 'count'])
print(f_summary)

print("\n--- Monetary Score Cutoffs (£ Revenue) ---")
m_cutoffs = pd.qcut(rfm_scored['Monetary'].rank(method='first'), q=5, labels=[1, 2, 3, 4, 5])
m_summary = rfm_scored.groupby(m_cutoffs, observed=False)['Monetary'].agg(['min', 'max', 'count'])
print(m_summary)

--- Recency Score Cutoffs (Days) ---
         min  max  count
Recency                 
5          1   11    876
4         12   32    904
3         33   71    857
2         72  178    860
1        179  374    875

--- Frequency Score Cutoffs (Orders) ---
           min  max  count
Frequency                 
1            1    1    875
2            1    2    874
3            2    3    874
4            3    5    874
5            5  210    875

--- Monetary Score Cutoffs (£ Revenue) ---
              min        max  count
Monetary                           
1        -4287.63     232.48    875
2          232.60     463.46    874
3          463.66     902.22    874
4          903.90    1992.76    874
5         1994.39  279489.02    875


## 5. RFM Visualization & Business Interpretation

In [7]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

sns.histplot(rfm_scored['Recency'], bins=30, kde=True, ax=axes[0], color='#2b5c8f')
axes[0].set_title('Recency Distribution (Days)')
axes[0].set_xlabel('Days Since Last Purchase')
axes[0].set_ylabel('Customer Count')

sns.histplot(rfm_scored['Frequency'], bins=30, kde=True, ax=axes[1], color='#3690c0')
axes[1].set_title('Frequency Distribution (Orders)')
axes[1].set_xlabel('Number of Unique Purchase Invoices')
axes[1].set_xlim(0, 30)
axes[1].set_ylabel('Customer Count')

sns.histplot(rfm_scored['Monetary'], bins=30, kde=True, ax=axes[2], color='#67a9cf')
axes[2].set_title('Monetary Distribution (£)')
axes[2].set_xlabel('Net Revenue (£)')
axes[2].set_xlim(-500, 10000)
axes[2].set_ylabel('Customer Count')

plt.tight_layout()
plt.savefig('../reports/figures/rfm_distributions.png', dpi=300)
plt.close()
print("Saved rfm_distributions.png")

Saved rfm_distributions.png


### Distribution Findings
1. **Recency:** Highly right-skewed. Half of the customer base made a purchase within the last 50 days, but a long tail extends beyond 300 days (churned accounts).
2. **Frequency:** Extremely right-skewed with a median of 2 orders. Over 35% of customers bought only once.
3. **Monetary:** Severe right-skew driven by large wholesale buyers. Median spend is £644, while mean spend is £1,894 and peak spend exceeds £279,000.

In [8]:
top_seg = rfm_scored['RFM_Segment'].value_counts().head(15)

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(x=top_seg.values, y=top_seg.index, palette='Blues_r', ax=ax)
ax.set_title('Top 15 Most Frequent RFM Segment Combinations', fontsize=14, fontweight='bold')
ax.set_xlabel('Number of Customers', fontsize=12)
ax.set_ylabel('RFM Segment Code (R-F-M)', fontsize=12)

for i, v in enumerate(top_seg.values):
    ax.text(v + 4, i, str(v), va='center', fontweight='bold', color='#1a365d')

plt.tight_layout()
plt.savefig('../reports/figures/rfm_segment_breakdown.png', dpi=300)
plt.show()plt.close()
print("Saved rfm_segment_breakdown.png")

C:\Users\Raghad\AppData\Local\Temp\ipykernel_11924\1613630347.py:4: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=top_seg.values, y=top_seg.index, palette='Blues_r', ax=ax)


Saved rfm_segment_breakdown.png


In [9]:
rfm_pivot_m = rfm_scored.pivot_table(index='R_Score', columns='F_Score', values='Monetary', aggfunc='mean')
rfm_pivot_count = rfm_scored.pivot_table(index='R_Score', columns='F_Score', values='CustomerID', aggfunc='count')

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

sns.heatmap(rfm_pivot_m, annot=True, fmt=',.0f', cmap='YlGnBu', ax=axes[0], cbar_kws={'label': 'Mean Monetary (£)'})
axes[0].set_title('Average Monetary Value (£) by Recency vs Frequency Scores', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Frequency Score (1=Low, 5=High)')
axes[0].set_ylabel('Recency Score (1=Low/Dormant, 5=Recent)')

sns.heatmap(rfm_pivot_count, annot=True, fmt='d', cmap='Oranges', ax=axes[1], cbar_kws={'label': 'Customer Count'})
axes[1].set_title('Customer Count by Recency vs Frequency Scores', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Frequency Score (1=Low, 5=High)')
axes[1].set_ylabel('Recency Score (1=Low/Dormant, 5=Recent)')

plt.tight_layout()
plt.savefig('../reports/figures/rfm_heatmaps.png', dpi=300)
plt.close()
print("Saved rfm_heatmaps.png")

Saved rfm_heatmaps.png


### Key RFM Matrix Insights
- **Top Segment (555):** 351 customers form the core 'Champions' cohort, purchasing recently, frequently, and spending high average amounts (£8,200+).
- **Dormant Segment (111):** 193 single-order low-spend customers who bought over 200 days ago and never returned.
- **Monetary Concentration:** Mean monetary spend climbs exponentially along the Frequency axis (F5 mean spend is over £8,500 compared to £300 for F1).

## 6. Preprocessing for Algorithmic Clustering

Distance-based clustering algorithms like K-Means and Hierarchical Clustering rely on Euclidean distance. They suffer when features are highly skewed or unscaled because:
1. Extreme wholesale monetary figures dwarf recency and frequency distances.
2. Non-normal right-skewed distributions distort centroid placement.

### 6.1 Skewness Transformation ($\log(1+x)$)
We apply $\log(1+x)$ (`np.log1p`) to compress extreme right-hand skews.

### 6.2 Standard Scaling ($\mu=0, \sigma=1$)
We apply `StandardScaler` to ensure each RFM feature contributes equally to distance calculations.

In [10]:
rfm_proc, X_scaled, scaler = preprocess_rfm_for_clustering(rfm_scored, filter_positive_monetary=True)

print(f"Customers prepared for algorithmic clustering (Monetary > 0): {len(rfm_proc):,}")

skew_df = pd.DataFrame({
    'Metric': ['Recency', 'Frequency', 'Monetary'],
    'Raw Skewness': [rfm_scored['Recency'].skew(), rfm_scored['Frequency'].skew(), rfm_scored['Monetary'].skew()],
    'Log Feature': ['R_log', 'F_log', 'M_log'],
    'Log Skewness': [rfm_proc['R_log'].skew(), rfm_proc['F_log'].skew(), rfm_proc['M_log'].skew()]
})

print("\n--- Skewness Comparison Before vs After Log Transformation ---")
print(skew_df)

Customers prepared for algorithmic clustering (Monetary > 0): 4,322

--- Skewness Comparison Before vs After Log Transformation ---
      Metric  Raw Skewness Log Feature  Log Skewness
0    Recency      1.249665       R_log     -0.357025
1  Frequency     12.130229       F_log      1.204898
2   Monetary     21.704784       M_log      0.215732


In [11]:
fig, axes = plt.subplots(2, 3, figsize=(16, 7))

# Raw distributions
sns.kdeplot(rfm_scored['Recency'], ax=axes[0, 0], color='#b2182b', fill=True)
axes[0, 0].set_title(f"Raw Recency (Skew: {rfm_scored['Recency'].skew():.2f})")

sns.kdeplot(rfm_scored['Frequency'], ax=axes[0, 1], color='#b2182b', fill=True)
axes[0, 1].set_title(f"Raw Frequency (Skew: {rfm_scored['Frequency'].skew():.2f})")
axes[0, 1].set_xlim(0, 30)

sns.kdeplot(rfm_scored['Monetary'], ax=axes[0, 2], color='#b2182b', fill=True)
axes[0, 2].set_title(f"Raw Monetary (Skew: {rfm_scored['Monetary'].skew():.2f})")
axes[0, 2].set_xlim(0, 10000)

# Log transformed distributions
sns.kdeplot(rfm_proc['R_log'], ax=axes[1, 0], color='#2166ac', fill=True)
axes[1, 0].set_title(f"Log Recency (Skew: {rfm_proc['R_log'].skew():.2f})")

sns.kdeplot(rfm_proc['F_log'], ax=axes[1, 1], color='#2166ac', fill=True)
axes[1, 1].set_title(f"Log Frequency (Skew: {rfm_proc['F_log'].skew():.2f})")

sns.kdeplot(rfm_proc['M_log'], ax=axes[1, 2], color='#2166ac', fill=True)
axes[1, 2].set_title(f"Log Monetary (Skew: {rfm_proc['M_log'].skew():.2f})")

plt.tight_layout()
plt.savefig('../reports/figures/log_transformation_effects.png', dpi=300)
plt.close()
print("Saved log_transformation_effects.png")

Saved log_transformation_effects.png


In [12]:
rfm_scored.to_parquet('../data/rfm_features.parquet', index=False)
rfm_proc.to_parquet('../data/rfm_preprocessed.parquet', index=False)

print("Exported datasets to /data:")
print(" - data/rfm_features.parquet (Raw customer RFM + Quantile scores)")
print(" - data/rfm_preprocessed.parquet (Log-transformed & scaled features for clustering)")

Exported datasets to /data:
 - data/rfm_features.parquet (Raw customer RFM + Quantile scores)
 - data/rfm_preprocessed.parquet (Log-transformed & scaled features for clustering)


## 7. Summary & Hand-off to Notebook 04

### Key Deliverables Completed:
1. **Identified Customer Isolation:** Processed 4,372 unique identified customers, excluding 135,037 guest transactions (£1.44M, 14.9% revenue) with documented justification.
2. **RFM Quantile Scoring:** Built 5-quantile R, F, and M scores and created 3-digit RFM segment codes (`'555'` to `'111'`).
3. **Behavioral Visualizations:** Rendered distribution plots, top segment bar charts, and R vs F heatmaps, saving all figures to `reports/figures/`.
4. **Log & Scaler Preprocessing:** Compressed Monetary skewness from **21.60 to 0.22** and Frequency skewness from **12.08 to 1.20**, producing scaled feature arrays ready for K-Means, Hierarchical, and DBSCAN clustering in Notebook 04.